<a href="https://colab.research.google.com/github/amzad-786githumb/AIR_LLM_Research/blob/main/03_Missingness_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# 03.1 LOAD PROCESSED TRAINING DATA
# ============================================================

from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib
import warnings

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Google Drive
# ------------------------------------------------------------

try:
    PROJECT_ROOT = Path(
        "/content/drive/MyDrive/AIR_LLM_Research"
    )

    if not PROJECT_ROOT.exists():
        raise FileNotFoundError(
            f"Project directory not found:\n{PROJECT_ROOT}"
        )

except Exception:
    drive.mount(
        "/content/drive",
        force_remount=False
    )

    PROJECT_ROOT = Path(
        "/content/drive/MyDrive/AIR_LLM_Research"
    )

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

SPLIT_ROOT = (
    PROJECT_ROOT /
    "data" /
    "splits"
)

MISSINGNESS_ROOT = (
    PROJECT_ROOT /
    "experiments" /
    "missingness"
)

GROUND_TRUTH_ROOT = (
    MISSINGNESS_ROOT /
    "ground_truth"
)

SCENARIO_ROOT = (
    MISSINGNESS_ROOT /
    "scenarios"
)

MASK_ROOT = (
    MISSINGNESS_ROOT /
    "masks"
)

for path in [
    MISSINGNESS_ROOT,
    GROUND_TRUTH_ROOT,
    SCENARIO_ROOT,
    MASK_ROOT,
]:
    path.mkdir(
        parents=True,
        exist_ok=True
    )

# ------------------------------------------------------------
# Dataset registry
# ------------------------------------------------------------

DATASET_IDS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]

TARGET_REGISTRY = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted",
}

# ------------------------------------------------------------
# Load training features
# ------------------------------------------------------------

TRAIN_FEATURES = {}

for dataset_id in DATASET_IDS:

    path = (
        SPLIT_ROOT /
        dataset_id /
        "X_train.csv"
    )

    if not path.exists():

        raise FileNotFoundError(
            f"Training features not found:\n{path}\n\n"
            "Run Notebook 02 successfully before Notebook 03."
        )

    df = pd.read_csv(
        path,
        low_memory=False
    )

    target = TARGET_REGISTRY[
        dataset_id
    ]

    # Safety check: target must not be in X_train.
    if target in df.columns:

        raise ValueError(
            f"Target '{target}' is present in X_train "
            f"for {dataset_id}. Target leakage detected."
        )

    TRAIN_FEATURES[
        dataset_id
    ] = df

    print(
        f"{dataset_id:<20} "
        f"rows={len(df):,} | "
        f"features={df.shape[1]:,}"
    )

print("=" * 90)
print("TRAINING DATA LOADED")
print("=" * 90)

Mounted at /content/drive
adult_income         rows=19,522 | features=14
bank_marketing       rows=27,126 | features=16
diabetes_130us       rows=61,059 | features=47
TRAINING DATA LOADED


In [2]:
# ============================================================
# 03.2 DEFINE MISSINGNESS MASKS
# ============================================================

MISSINGNESS_RATES = [
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
]

MISSINGNESS_MECHANISMS = [
    "MCAR",
    "MAR",
    "MNAR",
]

MASKING_MODES = [
    "cell_wise",
    "feature_wise",
]

RANDOM_SEEDS = [
    42,
    123,
    2025,
]

# ------------------------------------------------------------
# Mask conventions
# ------------------------------------------------------------
# True  = value is intentionally masked
# False = value remains observed
# ------------------------------------------------------------

def empty_mask(df):
    return pd.DataFrame(
        False,
        index=df.index,
        columns=df.columns
    )

def validate_mask(
    mask,
    df
):

    if not isinstance(
        mask,
        pd.DataFrame
    ):

        raise TypeError(
            "Mask must be a pandas DataFrame."
        )

    if mask.shape != df.shape:

        raise ValueError(
            "Mask and dataframe shapes do not match."
        )

    if list(mask.columns) != list(df.columns):

        raise ValueError(
            "Mask columns do not match dataframe columns."
        )

    return True

print("=" * 90)
print("MISSINGNESS CONFIGURATION")
print("=" * 90)

print(
    f"Mechanisms : {MISSINGNESS_MECHANISMS}"
)

print(
    f"Rates      : "
    f"{[int(x * 100) for x in MISSINGNESS_RATES]}%"
)

print(
    f"Modes      : {MASKING_MODES}"
)

print(
    f"Seeds      : {RANDOM_SEEDS}"
)

print("=" * 90)

MISSINGNESS CONFIGURATION
Mechanisms : ['MCAR', 'MAR', 'MNAR']
Rates      : [10, 20, 30, 40, 50]%
Modes      : ['cell_wise', 'feature_wise']
Seeds      : [42, 123, 2025]


In [3]:
# ============================================================
# 03.3 MCAR GENERATION
# ============================================================

def generate_mcar_mask(
    df,
    missing_rate,
    rng,
    eligible_columns=None
):
    """
    Generate a Missing Completely At Random (MCAR) mask.

    Each eligible cell has approximately equal probability
    of being selected, independently of observed values.
    """

    if not (
        0 < missing_rate < 1
    ):

        raise ValueError(
            "missing_rate must be between 0 and 1."
        )

    mask = empty_mask(df)

    if eligible_columns is None:

        eligible_columns = list(
            df.columns
        )

    n_rows = len(df)

    for column in eligible_columns:

        # Only currently observed values can be masked.
        observed = (
            ~df[column].isna()
        ).to_numpy()

        random_values = rng.random(
            n_rows
        )

        column_mask = (
            observed &
            (random_values < missing_rate)
        )

        mask[column] = column_mask

    return mask


print(
    "MCAR generator defined."
)

MCAR generator defined.


In [4]:
# ============================================================
# 03.4 MAR GENERATION
# ============================================================

def _numeric_driver_score(
    series
):
    """
    Convert a numerical observed feature into a normalized
    driver score in [0, 1].
    """

    values = pd.to_numeric(
        series,
        errors="coerce"
    )

    valid = values.notna()

    score = pd.Series(
        0.5,
        index=series.index,
        dtype=float
    )

    if valid.sum() <= 1:
        return score

    ranks = (
        values[valid]
        .rank(
            method="average",
            pct=True
        )
    )

    score.loc[valid] = ranks

    return score


def _categorical_driver_score(
    series
):
    """
    Convert a categorical observed feature into a normalized
    frequency-based driver score in [0, 1].
    """

    score = pd.Series(
        0.5,
        index=series.index,
        dtype=float
    )

    valid = series.notna()

    if valid.sum() == 0:
        return score

    frequencies = (
        series[valid]
        .value_counts(
            normalize=True
        )
    )

    frequency_score = (
        series[valid]
        .map(frequencies)
    )

    if frequency_score.max() == frequency_score.min():

        score.loc[valid] = 0.5

    else:

        score.loc[valid] = (
            1.0 -
            (
                frequency_score -
                frequency_score.min()
            ) /
            (
                frequency_score.max() -
                frequency_score.min()
            )
        )

    return score


def _select_mar_driver(
    df,
    target_column,
    candidate_columns
):

    """
    Select a deterministic observed feature as the MAR driver.

    Target is never used as a missingness driver.
    """

    candidates = [
        column
        for column in candidate_columns
        if column != target_column
    ]

    if not candidates:

        raise ValueError(
            "No valid MAR driver columns available."
        )

    # Prefer a numerical feature with sufficient variation.
    numerical_candidates = []

    for column in candidates:

        if pd.api.types.is_numeric_dtype(
            df[column]
        ):

            if df[column].nunique(
                dropna=True
            ) > 1:

                numerical_candidates.append(
                    column
                )

    if numerical_candidates:

        return numerical_candidates[0]

    return candidates[0]


def _calibrate_probability(
    score,
    target_rate,
    observed_mask
):
    """
    Calibrate a logistic probability so that the expected
    missingness rate approximately matches target_rate.
    """

    x = score.to_numpy(
        dtype=float
    )

    valid_x = x[observed_mask]

    if len(valid_x) == 0:
        return np.zeros(
            len(score),
            dtype=float
        )

    x = np.clip(
        x,
        0.0,
        1.0
    )

    # Binary search over intercept.
    low = -20.0
    high = 20.0

    for _ in range(60):

        intercept = (
            low + high
        ) / 2.0

        logits = (
            8.0 * (x - 0.5) +
            intercept
        )

        probability = (
            1.0 /
            (
                1.0 +
                np.exp(
                    -np.clip(
                        logits,
                        -30,
                        30
                    )
                )
            )
        )

        mean_probability = (
            probability[observed_mask]
            .mean()
        )

        if mean_probability > target_rate:
            high = intercept
        else:
            low = intercept

    return probability


def generate_mar_mask(
    df,
    missing_rate,
    rng,
    target_column=None,
    eligible_columns=None
):
    """
    Generate a Missing At Random (MAR) mask.

    Missingness of each target feature is driven by another
    observed feature. The target itself is never used as a
    driver.
    """

    mask = empty_mask(df)

    if eligible_columns is None:

        eligible_columns = list(
            df.columns
        )

    for column in eligible_columns:

        observed = (
            ~df[column].isna()
        ).to_numpy()

        drivers = [
            c
            for c in eligible_columns
            if c != column
        ]

        if not drivers:

            continue

        driver = _select_mar_driver(
            df,
            target_column,
            drivers
        )

        if pd.api.types.is_numeric_dtype(
            df[driver]
        ):

            score = _numeric_driver_score(
                df[driver]
            )

        else:

            score = _categorical_driver_score(
                df[driver]
            )

        probability = _calibrate_probability(
            score,
            missing_rate,
            observed
        )

        random_values = rng.random(
            len(df)
        )

        mask[column] = (
            observed &
            (
                random_values <
                probability
            )
        )

    return mask


print(
    "MAR generator defined."
)

MAR generator defined.


In [5]:
# ============================================================
# 03.5 MNAR GENERATION
# ============================================================

def _mnar_score(
    series
):
    """
    Construct a value-dependent MNAR score.

    Higher scores indicate greater probability of being
    masked. Numerical values use rank-based scores.
    Categorical values use category-frequency structure.
    """

    if pd.api.types.is_numeric_dtype(
        series
    ):

        return _numeric_driver_score(
            series
        )

    return _categorical_driver_score(
        series
    )


def generate_mnar_mask(
    df,
    missing_rate,
    rng,
    eligible_columns=None
):
    """
    Generate a Missing Not At Random (MNAR) mask.

    Missingness depends directly on the underlying value
    of the feature being masked.
    """

    mask = empty_mask(df)

    if eligible_columns is None:

        eligible_columns = list(
            df.columns
        )

    for column in eligible_columns:

        observed = (
            ~df[column].isna()
        ).to_numpy()

        score = _mnar_score(
            df[column]
        )

        probability = _calibrate_probability(
            score,
            missing_rate,
            observed
        )

        random_values = rng.random(
            len(df)
        )

        mask[column] = (
            observed &
            (
                random_values <
                probability
            )
        )

    return mask


print(
    "MNAR generator defined."
)

MNAR generator defined.


In [6]:
# ============================================================
# 03.6 10% MISSINGNESS
# ============================================================

RATE_10 = 0.10

print(
    f"10% missingness scenario configured: "
    f"{RATE_10:.0%}"
)

10% missingness scenario configured: 10%


In [7]:
# ============================================================
# 03.7 20% MISSINGNESS
# ============================================================

RATE_20 = 0.20

print(
    f"20% missingness scenario configured: "
    f"{RATE_20:.0%}"
)

20% missingness scenario configured: 20%


In [8]:
# ============================================================
# 03.8 30% MISSINGNESS
# ============================================================

RATE_30 = 0.30

print(
    f"30% missingness scenario configured: "
    f"{RATE_30:.0%}"
)

30% missingness scenario configured: 30%


In [9]:
# ============================================================
# 03.9 40% MISSINGNESS
# ============================================================

RATE_40 = 0.40

print(
    f"40% missingness scenario configured: "
    f"{RATE_40:.0%}"
)

40% missingness scenario configured: 40%


In [10]:
# ============================================================
# 03.10 50% MISSINGNESS
# ============================================================

RATE_50 = 0.50

print(
    f"50% missingness scenario configured: "
    f"{RATE_50:.0%}"
)

50% missingness scenario configured: 50%


In [11]:
# ============================================================
# 03.11 CELL-WISE MASKING
# ============================================================

def generate_cell_wise_mask(
    df,
    mechanism,
    missing_rate,
    seed,
    target_column=None
):

    rng = np.random.default_rng(
        seed
    )

    eligible_columns = list(
        df.columns
    )

    if target_column in eligible_columns:

        eligible_columns.remove(
            target_column
        )

    if mechanism == "MCAR":

        mask = generate_mcar_mask(
            df,
            missing_rate,
            rng,
            eligible_columns
        )

    elif mechanism == "MAR":

        mask = generate_mar_mask(
            df,
            missing_rate,
            rng,
            target_column,
            eligible_columns
        )

    elif mechanism == "MNAR":

        mask = generate_mnar_mask(
            df,
            missing_rate,
            rng,
            eligible_columns
        )

    else:

        raise ValueError(
            f"Unsupported mechanism: {mechanism}"
        )

    # Target must never be masked.
    if target_column is not None:

        if target_column in mask.columns:

            mask[
                target_column
            ] = False

    validate_mask(
        mask,
        df
    )

    return mask


print(
    "Cell-wise masking interface defined."
)

Cell-wise masking interface defined.


In [12]:
# ============================================================
# 03.12 FEATURE-WISE MASKING
# ============================================================

def generate_feature_wise_mask(
    df,
    mechanism,
    missing_rate,
    seed,
    target_column=None
):
    """
    Feature-wise masking.

    A controlled subset of feature columns is selected, and
    missingness is generated within those selected features.

    This preserves the distinction between:
        cell-wise missingness
        feature-wise missingness
    """

    rng = np.random.default_rng(
        seed
    )

    eligible_columns = [
        column
        for column in df.columns
        if column != target_column
    ]

    if not eligible_columns:

        raise ValueError(
            "No eligible features available."
        )

    n_features = len(
        eligible_columns
    )

    # Select approximately the requested proportion
    # of features while guaranteeing at least one.
    n_selected = max(
        1,
        int(
            np.ceil(
                n_features *
                missing_rate
            )
        )
    )

    n_selected = min(
        n_selected,
        n_features
    )

    selected_columns = rng.choice(
        eligible_columns,
        size=n_selected,
        replace=False
    )

    mask = empty_mask(
        df
    )

    if mechanism == "MCAR":

        for column in selected_columns:

            observed = (
                ~df[column].isna()
            ).to_numpy()

            random_values = rng.random(
                len(df)
            )

            mask[column] = (
                observed &
                (
                    random_values <
                    missing_rate
                )
            )

    elif mechanism == "MAR":

        for column in selected_columns:

            observed = (
                ~df[column].isna()
            ).to_numpy()

            drivers = [
                c
                for c in eligible_columns
                if c != column
            ]

            driver = _select_mar_driver(
                df,
                target_column,
                drivers
            )

            if pd.api.types.is_numeric_dtype(
                df[driver]
            ):

                score = _numeric_driver_score(
                    df[driver]
                )

            else:

                score = _categorical_driver_score(
                    df[driver]
                )

            probability = _calibrate_probability(
                score,
                missing_rate,
                observed
            )

            random_values = rng.random(
                len(df)
            )

            mask[column] = (
                observed &
                (
                    random_values <
                    probability
                )
            )

    elif mechanism == "MNAR":

        for column in selected_columns:

            observed = (
                ~df[column].isna()
            ).to_numpy()

            score = _mnar_score(
                df[column]
            )

            probability = _calibrate_probability(
                score,
                missing_rate,
                observed
            )

            random_values = rng.random(
                len(df)
            )

            mask[column] = (
                observed &
                (
                    random_values <
                    probability
                )
            )

    else:

        raise ValueError(
            f"Unsupported mechanism: {mechanism}"
        )

    if target_column is not None:

        if target_column in mask.columns:

            mask[
                target_column
            ] = False

    validate_mask(
        mask,
        df
    )

    return mask, list(
        selected_columns
    )


print(
    "Feature-wise masking interface defined."
)

Feature-wise masking interface defined.


In [13]:
# ============================================================
# 03.13 MULTIPLE RANDOM SEEDS
# ============================================================

EXPERIMENT_DESIGN = []

for dataset_id in DATASET_IDS:

    for mechanism in MISSINGNESS_MECHANISMS:

        for rate in MISSINGNESS_RATES:

            for mode in MASKING_MODES:

                for seed in RANDOM_SEEDS:

                    EXPERIMENT_DESIGN.append({

                        "dataset_id":
                            dataset_id,

                        "mechanism":
                            mechanism,

                        "missing_rate":
                            float(rate),

                        "missing_rate_percent":
                            int(rate * 100),

                        "masking_mode":
                            mode,

                        "seed":
                            int(seed),
                    })

EXPERIMENT_DESIGN_DF = pd.DataFrame(
    EXPERIMENT_DESIGN
)

EXPERIMENT_DESIGN_DF[
    "scenario_id"
] = EXPERIMENT_DESIGN_DF.apply(
    lambda row:
        (
            f"{row['dataset_id']}_"
            f"{row['mechanism'].lower()}_"
            f"{row['masking_mode']}_"
            f"{row['missing_rate_percent']}pct_"
            f"seed{row['seed']}"
        ),
    axis=1
)

print("=" * 90)
print("EXPERIMENTAL DESIGN")
print("=" * 90)

print(
    f"Total scenarios: "
    f"{len(EXPERIMENT_DESIGN_DF):,}"
)

print(
    f"Datasets       : {len(DATASET_IDS)}"
)

print(
    f"Mechanisms     : {len(MISSINGNESS_MECHANISMS)}"
)

print(
    f"Rates          : {len(MISSINGNESS_RATES)}"
)

print(
    f"Masking modes  : {len(MASKING_MODES)}"
)

print(
    f"Random seeds   : {len(RANDOM_SEEDS)}"
)

print("=" * 90)

EXPERIMENTAL DESIGN
Total scenarios: 270
Datasets       : 3
Mechanisms     : 3
Rates          : 5
Masking modes  : 2
Random seeds   : 3


In [14]:
# ============================================================
# 03.14 GROUND-TRUTH PRESERVATION
# ============================================================

GROUND_TRUTH_METADATA = {}

for dataset_id in DATASET_IDS:

    df = TRAIN_FEATURES[
        dataset_id
    ].copy()

    output_path = (
        GROUND_TRUTH_ROOT /
        f"{dataset_id}_X_train_ground_truth.parquet"
    )

    try:

        df.to_parquet(
            output_path,
            index=False
        )

    except Exception:

        # Fallback for environments without pyarrow.
        output_path = (
            GROUND_TRUTH_ROOT /
            f"{dataset_id}_X_train_ground_truth.csv"
        )

        df.to_csv(
            output_path,
            index=False
        )

    # Dataset fingerprint for reproducibility.
    fingerprint = hashlib.sha256(
        pd.util.hash_pandas_object(
            df,
            index=True
        ).values.tobytes()
    ).hexdigest()

    GROUND_TRUTH_METADATA[
        dataset_id
    ] = {

        "rows":
            int(df.shape[0]),

        "columns":
            int(df.shape[1]),

        "column_names":
            list(df.columns),

        "fingerprint_sha256":
            fingerprint,

        "ground_truth_path":
            str(output_path),
    }

with open(
    GROUND_TRUTH_ROOT /
    "ground_truth_metadata.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        GROUND_TRUTH_METADATA,
        f,
        indent=2
    )

print("=" * 90)
print("GROUND TRUTH PRESERVED")
print("=" * 90)

for dataset_id, metadata in (
    GROUND_TRUTH_METADATA.items()
):

    print(
        f"{dataset_id:<20}"
        f"rows={metadata['rows']:,} | "
        f"columns={metadata['columns']:,}"
    )

print("=" * 90)

GROUND TRUTH PRESERVED
adult_income        rows=19,522 | columns=14
bank_marketing      rows=27,126 | columns=16
diabetes_130us      rows=61,059 | columns=47


In [15]:
# ============================================================
# 03.15 MISSINGNESS VALIDATION
# ============================================================

def calculate_mask_statistics(
    original_df,
    masked_df,
    mask,
    dataset_id,
    mechanism,
    rate,
    mode,
    seed,
    scenario_id
):

    validate_mask(
        mask,
        original_df
    )

    feature_columns = list(
        original_df.columns
    )

    total_eligible_cells = 0
    total_masked_cells = 0

    for column in feature_columns:

        observed = (
            ~original_df[column].isna()
        )

        total_eligible_cells += int(
            observed.sum()
        )

        total_masked_cells += int(
            (
                mask[column] &
                observed
            ).sum()
        )

    achieved_rate = (
        total_masked_cells /
        total_eligible_cells
        if total_eligible_cells > 0
        else 0.0
    )

    # Ensure all masked cells became NaN.
    mask_application_correct = True

    for column in feature_columns:

        masked_positions = (
            mask[column]
        )

        if (
            masked_df.loc[
                masked_positions,
                column
            ].notna().any()
        ):

            mask_application_correct = False
            break

    # Ensure target is never masked.
    target = TARGET_REGISTRY[
        dataset_id
    ]

    target_protected = (
        target not in mask.columns or
        not mask[target].any()
    )

    return {

        "scenario_id":
            scenario_id,

        "dataset_id":
            dataset_id,

        "mechanism":
            mechanism,

        "masking_mode":
            mode,

        "requested_rate":
            float(rate),

        "achieved_rate":
            float(achieved_rate),

        "rate_error":
            float(
                abs(
                    achieved_rate -
                    rate
                )
            ),

        "seed":
            int(seed),

        "total_rows":
            int(len(original_df)),

        "total_features":
            int(original_df.shape[1]),

        "eligible_cells":
            int(total_eligible_cells),

        "masked_cells":
            int(total_masked_cells),

        "mask_application_correct":
            bool(mask_application_correct),

        "target_protected":
            bool(target_protected),
    }


VALIDATION_RESULTS = []

for row in EXPERIMENT_DESIGN_DF.itertuples(
    index=False
):

    dataset_id = row.dataset_id
    mechanism = row.mechanism
    rate = float(row.missing_rate)
    mode = row.masking_mode
    seed = int(row.seed)
    scenario_id = row.scenario_id

    original_df = TRAIN_FEATURES[
        dataset_id
    ].copy()

    target = TARGET_REGISTRY[
        dataset_id
    ]

    if mode == "cell_wise":

        mask = generate_cell_wise_mask(
            original_df,
            mechanism,
            rate,
            seed,
            target
        )

        selected_features = list(
            original_df.columns
        )

    elif mode == "feature_wise":

        mask, selected_features = (
            generate_feature_wise_mask(
                original_df,
                mechanism,
                rate,
                seed,
                target
            )
        )

    else:

        raise ValueError(
            f"Unknown masking mode: {mode}"
        )

    # --------------------------------------------------------
    # Apply mask without modifying original data.
    # --------------------------------------------------------

    masked_df = original_df.copy()

    masked_df = masked_df.mask(
        mask
    )

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    statistics = calculate_mask_statistics(
        original_df,
        masked_df,
        mask,
        dataset_id,
        mechanism,
        rate,
        mode,
        seed,
        scenario_id
    )

    statistics[
        "selected_feature_count"
    ] = int(
        len(selected_features)
    )

    statistics[
        "selected_features"
    ] = json.dumps(
        selected_features
    )

    VALIDATION_RESULTS.append(
        statistics
    )

VALIDATION_DF = pd.DataFrame(
    VALIDATION_RESULTS
)

# ------------------------------------------------------------
# Validation assertions
# ------------------------------------------------------------

if not (
    VALIDATION_DF[
        "mask_application_correct"
    ].all()
):

    raise AssertionError(
        "At least one scenario failed mask application validation."
    )

if not (
    VALIDATION_DF[
        "target_protected"
    ].all()
):

    raise AssertionError(
        "Target leakage detected: target was masked."
    )

print("=" * 90)
print("MISSINGNESS VALIDATION COMPLETE")
print("=" * 90)

print(
    f"Scenarios validated: "
    f"{len(VALIDATION_DF):,}"
)

print(
    f"Minimum achieved rate: "
    f"{VALIDATION_DF['achieved_rate'].min():.4f}"
)

print(
    f"Maximum achieved rate: "
    f"{VALIDATION_DF['achieved_rate'].max():.4f}"
)

print(
    f"Maximum rate error: "
    f"{VALIDATION_DF['rate_error'].max():.4f}"
)

print("=" * 90)

display(
    VALIDATION_DF.head(10)
)

MISSINGNESS VALIDATION COMPLETE
Scenarios validated: 270
Minimum achieved rate: 0.0071
Maximum achieved rate: 0.5019
Maximum rate error: 0.2595


,scenario_id,dataset_id,mechanism,masking_mode,requested_rate,achieved_rate,rate_error,seed,total_rows,total_features,eligible_cells,masked_cells,mask_application_correct,target_protected,selected_feature_count,selected_features
0,adult_income_mcar_cell_wise_10pct_seed42,adult_income,MCAR,cell_wise,0.1,0.099371,0.000629,42,19522,14,273308,27159,True,True,14,"[""age"", ""workclass"", ""fnlwgt"", ""education"", ""e..."
1,adult_income_mcar_cell_wise_10pct_seed123,adult_income,MCAR,cell_wise,0.1,0.099986,0.000014,123,19522,14,273308,27327,True,True,14,"[""age"", ""workclass"", ""fnlwgt"", ""education"", ""e..."
2,adult_income_mcar_cell_wise_10pct_seed2025,adult_income,MCAR,cell_wise,0.1,0.100824,0.000824,2025,19522,14,273308,27556,True,True,14,"[""age"", ""workclass"", ""fnlwgt"", ""education"", ""e..."
3,adult_income_mcar_feature_wise_10pct_seed42,adult_income,MCAR,feature_wise,0.1,0.014046,0.085954,42,19522,14,273308,3839,True,True,2,"[""workclass"", ""capital_gain""]"
4,adult_income_mcar_feature_wise_10pct_seed123,adult_income,MCAR,feature_wise,0.1,0.014226,0.085774,123,19522,14,273308,3888,True,True,2,"[""age"", ""sex""]"
5,adult_income_mcar_feature_wise_10pct_seed2025,adult_income,MCAR,feature_wise,0.1,0.014401,0.085599,2025,19522,14,273308,3936,True,True,2,"[""marital_status"", ""native_country""]"
6,adult_income_mcar_cell_wise_20pct_seed42,adult_income,MCAR,cell_wise,0.2,0.200137,0.000137,42,19522,14,273308,54699,True,True,14,"[""age"", ""workclass"", ""fnlwgt"", ""education"", ""e..."
7,adult_income_mcar_cell_wise_20pct_seed123,adult_income,MCAR,cell_wise,0.2,0.199215,0.000785,123,19522,14,273308,54447,True,True,14,"[""age"", ""workclass"", ""fnlwgt"", ""education"", ""e..."
8,adult_income_mcar_cell_wise_20pct_seed2025,adult_income,MCAR,cell_wise,0.2,0.200872,0.000872,2025,19522,14,273308,54900,True,True,14,"[""age"", ""workclass"", ""fnlwgt"", ""education"", ""e..."
9,adult_income_mcar_feature_wise_20pct_seed42,adult_income,MCAR,feature_wise,0.2,0.042849,0.157151,42,19522,14,273308,11711,True,True,3,"[""sex"", ""workclass"", ""capital_gain""]"


In [16]:
# ============================================================
# 03.16 SAVE EXPERIMENTAL SCENARIOS
# ============================================================

SCENARIO_INDEX = []

for row in EXPERIMENT_DESIGN_DF.itertuples(
    index=False
):

    dataset_id = row.dataset_id
    mechanism = row.mechanism
    rate = float(row.missing_rate)
    mode = row.masking_mode
    seed = int(row.seed)
    scenario_id = row.scenario_id

    original_df = TRAIN_FEATURES[
        dataset_id
    ].copy()

    target = TARGET_REGISTRY[
        dataset_id
    ]

    # --------------------------------------------------------
    # Generate mask
    # --------------------------------------------------------

    if mode == "cell_wise":

        mask = generate_cell_wise_mask(
            original_df,
            mechanism,
            rate,
            seed,
            target
        )

        selected_features = list(
            original_df.columns
        )

    else:

        mask, selected_features = (
            generate_feature_wise_mask(
                original_df,
                mechanism,
                rate,
                seed,
                target
            )
        )

    # --------------------------------------------------------
    # Apply mask
    # --------------------------------------------------------

    masked_df = original_df.mask(
        mask
    )

    # --------------------------------------------------------
    # Scenario directories
    # --------------------------------------------------------

    scenario_dir = (
        SCENARIO_ROOT /
        dataset_id /
        mechanism.lower() /
        mode /
        f"{int(rate * 100)}pct" /
        f"seed_{seed}"
    )

    scenario_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    # --------------------------------------------------------
    # Save masked dataset
    # --------------------------------------------------------

    masked_path = (
        scenario_dir /
        "masked_data.parquet"
    )

    try:

        masked_df.to_parquet(
            masked_path,
            index=False
        )

    except Exception:

        masked_path = (
            scenario_dir /
            "masked_data.csv.gz"
        )

        masked_df.to_csv(
            masked_path,
            index=False,
            compression="gzip"
        )

    # --------------------------------------------------------
    # Save mask
    # --------------------------------------------------------

    mask_array = mask.to_numpy(
        dtype=np.uint8
    )

    mask_path = (
        scenario_dir /
        "missingness_mask.npz"
    )

    np.savez_compressed(
        mask_path,
        mask=mask_array,
        columns=np.array(
            mask.columns,
            dtype=str
        )
    )

    # --------------------------------------------------------
    # Save scenario metadata
    # --------------------------------------------------------

    achieved_rate = (
        VALIDATION_DF.loc[
            VALIDATION_DF[
                "scenario_id"
            ] == scenario_id,
            "achieved_rate"
        ].iloc[0]
    )

    metadata = {

        "scenario_id":
            scenario_id,

        "dataset_id":
            dataset_id,

        "mechanism":
            mechanism,

        "masking_mode":
            mode,

        "requested_missing_rate":
            rate,

        "achieved_missing_rate":
            float(achieved_rate),

        "seed":
            seed,

        "target_column":
            target,

        "target_masked":
            False,

        "selected_features":
            selected_features,

        "source_training_data":
            str(
                SPLIT_ROOT /
                dataset_id /
                "X_train.csv"
            ),

        "ground_truth":
            str(
                GROUND_TRUTH_ROOT /
                f"{dataset_id}_X_train_ground_truth.parquet"
            ),

        "masked_dataset":
            str(masked_path),

        "mask":
            str(mask_path),
    }

    metadata_path = (
        scenario_dir /
        "scenario_metadata.json"
    )

    with open(
        metadata_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            metadata,
            f,
            indent=2
        )

    SCENARIO_INDEX.append(
        metadata
    )

# ------------------------------------------------------------
# Save scenario index
# ------------------------------------------------------------

SCENARIO_INDEX_DF = pd.DataFrame(
    SCENARIO_INDEX
)

INDEX_PATH = (
    MISSINGNESS_ROOT /
    "experimental_scenario_index.csv"
)

SCENARIO_INDEX_DF.to_csv(
    INDEX_PATH,
    index=False
)

# ------------------------------------------------------------
# Save validation report
# ------------------------------------------------------------

VALIDATION_PATH = (
    MISSINGNESS_ROOT /
    "missingness_validation.csv"
)

VALIDATION_DF.to_csv(
    VALIDATION_PATH,
    index=False
)

# ------------------------------------------------------------
# Save experimental design
# ------------------------------------------------------------

DESIGN_PATH = (
    MISSINGNESS_ROOT /
    "experimental_design.csv"
)

EXPERIMENT_DESIGN_DF.to_csv(
    DESIGN_PATH,
    index=False
)

print("=" * 90)
print("EXPERIMENTAL SCENARIOS SAVED")
print("=" * 90)

print(
    f"Total scenarios : "
    f"{len(SCENARIO_INDEX_DF):,}"
)

print(
    f"Scenario root   : "
    f"{SCENARIO_ROOT}"
)

print(
    f"Scenario index  : "
    f"{INDEX_PATH}"
)

print(
    f"Validation file : "
    f"{VALIDATION_PATH}"
)

print(
    f"Design file     : "
    f"{DESIGN_PATH}"
)

print("=" * 90)

EXPERIMENTAL SCENARIOS SAVED
Total scenarios : 270
Scenario root   : /content/drive/MyDrive/AIR_LLM_Research/experiments/missingness/scenarios
Scenario index  : /content/drive/MyDrive/AIR_LLM_Research/experiments/missingness/experimental_scenario_index.csv
Validation file : /content/drive/MyDrive/AIR_LLM_Research/experiments/missingness/missingness_validation.csv
Design file     : /content/drive/MyDrive/AIR_LLM_Research/experiments/missingness/experimental_design.csv
